## NeuroPath

In [22]:
import torch
import torch.nn as nn
class DetourTransformer(nn.Module):

    def __init__(self, 
        heads: int = 2,
        nlayer: int = 1,
        node_sz: int=116,
        in_channel = 10,
        out_channel: int = 10,
        dropout: float = 0.1,
        hiddim: int = 1024,
        batch_size = 32,
        device='cuda:0',
        lconsist_w=1,
        *args, **kwargs) -> None:
        
        super(DetourTransformer, self).__init__()
        self.lconsist_w = lconsist_w
        org_in_channel = in_channel
        # in_channel = out_channel
        if in_channel % heads != 0:
            in_channel = in_channel  + heads - (in_channel % heads)
        self.nlayer = nlayer
        self.node_sz = node_sz
            
        self.lin_first = nn.Sequential(
            nn.Linear(org_in_channel, in_channel), 
            nn.BatchNorm1d(in_channel), 
            nn.LeakyReLU()
        )
        self.lin_in = nn.Sequential(
            nn.Linear(in_channel, out_channel), 
            nn.BatchNorm1d(out_channel), 
            nn.LeakyReLU(),
        )
        # self.net = torch.nn.TransformerEncoder(
        #     torch.nn.TransformerEncoderLayer(d_model=in_channel, nhead=heads, dim_feedforward=in_channel, dropout=dropout, batch_first=True),
        #     num_layers=nlayer,
        #     norm=None#nn.LayerNorm(in_channel)
        # )
        self.net = nn.ModuleList([torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(d_model=in_channel, nhead=heads, dim_feedforward=hiddim, dropout=dropout, batch_first=True),
            num_layers=1,
            norm=None# #nn.LayerNorm(in_channel) # None#
        ) for _ in range(nlayer)])
        self.net_fc = nn.ModuleList([torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(d_model=in_channel, nhead=heads, dim_feedforward=hiddim, dropout=dropout, batch_first=True),
            num_layers=1,
            norm=None# #nn.LayerNorm(in_channel) # None#
        ) for _ in range(nlayer)])
        self.heads = heads
        self.in_channel = in_channel
        self.out_channel = out_channel
        self.mask_heldout = torch.zeros(batch_size, node_sz, node_sz) - torch.inf
        self.mask_heldout = self.mask_heldout.to(device)
        self.fcsc_loss = nn.MSELoss()
        ###################################
        self.loss = 0
        ###################################

    def forward(self, data):
        self.loss = 0
        node_feature = data.x
        node_feature = self.lin_first(node_feature)
        node_feature = node_feature.view(data.batch.max()+1, len(torch.where(data.batch==0)[0]), self.in_channel)
        ###################################
        node_feature_fc = node_feature
        ###################################

        adj = data.adj_sc
        adj_fc = data.adj_fc
        org_adj = adj
        multi_mask = []
        for _ in range(self.heads):
            if self.mask_heldout.shape[1] != adj.shape[1]:
                self.mask_heldout = torch.zeros(self.mask_heldout.shape[0], adj.shape[1], adj.shape[2], device=adj.device) - torch.inf
            mask = self.mask_heldout[:len(adj)]
            mask[torch.logical_and(adj, adj_fc)] = 0
            adj = (adj.float() @ org_adj.float()) > 0
            multi_mask.append(mask)
        multi_mask = torch.cat(multi_mask)
        # multi_mask = multi_mask==0
        ### fmask(FC) #################################
        mask_fc = self.mask_heldout[:len(adj_fc)]
        mask_fc[adj_fc] = 0
        mask_fc = mask_fc.repeat(self.heads, 1, 1)
        # mask_fc = mask_fc==0
        ###############################################
        for i in range(self.nlayer):
            node_feature = self.net[i](node_feature, mask=multi_mask) + node_feature
        ## readout feature ############################
            node_feature_fc = self.net_fc[i](node_feature_fc, mask=mask_fc)
            self.loss = self.loss + self.lconsist_w*self.fcsc_loss(node_feature_fc, node_feature)
        if not self.training:
            node_feature = node_feature_fc
        ###############################################
        return self.lin_in(node_feature.reshape(node_feature.shape[0] * node_feature.shape[1], self.in_channel))


class GraphNet(nn.Module):
    def __init__(self, in_channel, out_channel, hid_channel=768, **kwargs) -> None:
        super().__init__()
        self.node_sz = 116
        self.net = DetourTransformer(node_sz=self.node_sz, in_channel=in_channel, out_channel=hid_channel, hiddim=hid_channel, nlayer=4, heads=8, batch_size=batch_size, device=device)
        self.lin_node = nn.Linear(self.node_sz, 1)
        self.lin_out = nn.Linear(hid_channel, out_channel)

    def forward(self, batch):
        x = self.net(batch)
        x = torch.stack(x.split(self.node_sz)).permute(0, 2, 1)
        x = self.lin_node(x)[..., 0]
        return self.lin_out(x)

        

In [4]:
import numpy as np
# source: https://www.medrxiv.org/content/medrxiv/early/2022/02/17/2022.02.16.22271085/DC1/embed/media-1.pdf?download=true
brain_code_str='''A81 
E71, E75, E75, E75 
F01, F02, F03, F04, F10, F10,
F10, F10, F13, F13, F13,
F13, F18, F18
G10, G20, G30, G23, G31, G31
R41'''
brain_codes = []
for line in brain_code_str.split('\n'):
    brain_codes.extend([i for i in line.split(', ')])
brain_codes = np.unique(brain_codes).tolist()
print(brain_codes)
# source: https://www.cms.gov/medicare/coding/icd10/downloads/icd10clinicalconceptscardiology1.pdf
heart_codes = ['R00', 'I48', 'I49', 'I20', 'R07', 'I34', 'I35', 'I10', 'I50', 'I21', 'I22', 'I23', 'I25', 'R55']
kidney_codes = [f'N{i:02d}' for i in range(100)]
liver_codes = [f'K{i:02d}' for i in range(70, 78, 1)]

['A81 ', 'E71', 'E75', 'E75 ', 'F01', 'F02', 'F03', 'F04', 'F10', 'F10,', 'F13', 'F13,', 'F18', 'G10', 'G20', 'G23', 'G30', 'G31', 'R41']


In [23]:
def multiclass_eval(classifier, device, xs_tensor, ys_tensor, return_pred=False):
    # classifier.to(device)
    classifier.eval()
    y_true = []
    y_pred = []
    y_scores = []
    losses = []
    loss_fn = nn.CrossEntropyLoss()
    # for bi in range(0, len(val_idx), batch_size):
    for bi in range(0, len(xs_tensor), batch_size):
        x, y = xs_tensor[bi : bi+batch_size].to(device), ys_tensor[bi : bi+batch_size].to(device)
        batch = {'x': x.reshape(len(x), -1).float(), 'y': y}
        with torch.no_grad():
            y = classifier(batch) 
            loss = loss_fn(y.float(), batch['y'].long())
        if return_pred:
            # y_scores.append(y[torch.arange(len(y)), batch['y']].detach().cpu())
            y_scores.append(y.detach().cpu())
            y_pred.append(y.argmax(1).detach().cpu())
            y_true.append(batch['y'])
        losses.append(loss.detach().cpu().item())

    if return_pred:
        y_true = torch.cat(y_true, dim = 0).detach().cpu().numpy()
        y_pred = torch.cat(y_pred, dim = 0).detach().cpu().numpy()
        y_scores = torch.cat(y_scores, dim = 0).detach().cpu().numpy()
        return np.mean(losses), y_true, y_pred, y_scores
    else:
        return np.mean(losses)


In [32]:
from dataset.icdfc_dataset import ICDFCDataset
import torchvision
# import argparse
import yaml
import os
from torchvision.utils import make_grid
from tqdm import tqdm
from models.unet_cond_base import Unet
from scheduler.linear_noise_scheduler import LinearNoiseScheduler
from utils.config_utils import *


config_path = 'config/icd2fc_image_cond.yaml'
# Read the config file #
with open(config_path, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)
print(config)
########################

diffusion_config = config['diffusion_params']
dataset_config = config['dataset_params']
# dataset_config['im_size'] = 120
diffusion_model_config = config['ldm_params']
autoencoder_model_config = config['autoencoder_params']
train_config = config['train_params']
diffusion_config['num_timesteps'] = 100

########## Create the noise scheduler #############
scheduler = LinearNoiseScheduler(num_timesteps=diffusion_config['num_timesteps'],
                                 beta_start=diffusion_config['beta_start'],
                                 beta_end=diffusion_config['beta_end'])
###############################################

# text_tokenizer = None

############# Validate the config #################
condition_config = get_config_value(diffusion_model_config, key='condition_config', default_value=None)
assert condition_config is not None, ("This sampling script is for text conditional "
                                      "but no conditioning config found")
condition_types = get_config_value(condition_config, 'condition_types', [])
assert 'text' in condition_types, ("This sampling script is for text conditional "
                                    "but no text condition found in config")
validate_text_config(condition_config)
###############################################
latent_tag = 'DualSPD_thr25'

im_dataset = ICDFCDataset(split='val', preload_embed=True, device='cuda:1', skip_future=False,
                            im_path=dataset_config['im_path'],
                            im_size=dataset_config['im_size'],
                            im_channels=dataset_config['im_channels'],
                            use_latents=True,
                            latent_path=os.path.join(train_config['task_name'],
                                                     latent_tag+train_config['vqvae_latent_dir_name']),
                            condition_config=condition_config)
im_dataset.image_names[:10]

{'dataset_params': {'im_path': 'placeholder', 'im_channels': 1, 'im_size': 116, 'name': 'ICDFC'}, 'diffusion_params': {'num_timesteps': 1000, 'beta_start': 0.00085, 'beta_end': 0.012}, 'ldm_params': {'down_channels': [128, 256, 384, 512], 'mid_channels': [512, 384], 'down_sample': [True, True, True], 'attn_down': [True, True, True], 'time_emb_dim': 512, 'norm_channels': 32, 'num_heads': 8, 'conv_out_channels': 128, 'num_down_layers': 1, 'num_mid_layers': 2, 'num_up_layers': 1, 'condition_config': {'condition_types': ['text'], 'text_condition_config': {'text_embed_model': 'clip', 'train_text_embed_model': False, 'text_embed_dim': 512, 'cond_drop_prob': 0.1}, 'image_condition_config': {'image_condition_input_channels': 1, 'image_condition_output_channels': 1, 'image_condition_h': 120, 'image_condition_w': 120, 'cond_drop_prob': 0.1}}}, 'autoencoder_params': {'z_channels': 1, 'codebook_size': 8192, 'down_channels': [64, 128, 256, 256], 'mid_channels': [256, 256], 'down_sample': [True, Tru

prepare data list: 100%|████████████████| 19930/19930 [00:03<00:00, 5266.05it/s]


Found 4485 latents


['1000060-0',
 '1002138-1',
 '1003157-1',
 '1004050-0',
 '1005250-0',
 '1005474-1',
 '1005474-3',
 '1005841-1',
 '1005888-1',
 '1005888-3']

In [8]:
val = torch.load('../StableDiffusion-PyTorch/ntp_real-fc_val.pth')
val_data = {'gen_img': val['gt_img']}
val_nd = val['next_disease']
train = torch.load('../StableDiffusion-PyTorch/ntp_real-fc_train.pth')
all_train_data = {'gen_img': train['gt_img']}
train_nd = train['next_disease']
uni_nd = np.unique(np.concatenate([np.unique(train_nd), np.unique(val_nd)]))
train_labelid = np.stack([uni_nd==i for i in train_nd]).astype(float)
val_labelid = np.stack([uni_nd==i for i in val_nd]).astype(float)

In [39]:
dataid = torch.load('ntp_text_embed_val.pth')['data_id']
val_eidlist = [im_dataset.image_names[i].split('-')[0] for i in dataid]

In [96]:
dataid = torch.load('ntp_text_embed_train.pth')['data_id']
im_dataset = ICDFCDataset(split='train', preload_embed=True, device='cuda:1', skip_future=False,
                            im_path=dataset_config['im_path'],
                            im_size=dataset_config['im_size'],
                            im_channels=dataset_config['im_channels'],
                            use_latents=True,
                            latent_path=os.path.join(train_config['task_name'],
                                                     latent_tag+train_config['vqvae_latent_dir_name']),
                            condition_config=condition_config)

train_eidlist = [im_dataset.image_names[i].split('-')[0] for i in dataid]

['hariri', 'rest']


prepare data list: 100%|████████████████| 19930/19930 [00:13<00:00, 1469.62it/s]


Found 17932 latents


In [ ]:
import pandas as pd
data = pd.read_pickle('../neuro_detour/data/ukb-nimg-dwi_icd10_dated.pkl')
data

In [97]:
val_sc = data.loc[np.array(val_eidlist).astype(int)]['SC']
train_sc = data.loc[np.array(train_eidlist).astype(int)]['SC']

val_sc[~val_sc.isna()].shape, train_sc[~train_sc.isna()].shape

((6221,), (23208,))

array([    0,     1,     2, ..., 10698, 10699, 10701], shape=(4488,))

In [9]:
np_softmax = lambda x: np.exp(x - np.max(x, -1)[:, None])

In [43]:
from torch_geometric.data import Batch, Data


In [57]:
brain_code

'F10'

In [60]:
val_batches[0]

DataBatch(x=[3840, 120], edge_index=[2, 24422], y=[32], adj_sc=[3712, 116], adj_fc=[32, 120, 120], batch=[3840], ptr=[33])

In [89]:
import random
from tqdm import trange
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_curve, auc

 
import warnings
warnings.filterwarnings('ignore')

def multiclass_train(classifier, device, loader, optimizer, epoch):
    classifier.train()
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for batch in loader:
        batch = batch.to(device)
        gt = batch['y']
        optimizer.zero_grad()
        assert not batch['x'].isnan().any() and not batch['x'].isinf().any(), batch['x']
        y = classifier(batch) 
        loss = loss_fn(y.float(), gt.long()) + classifier.net.loss
        loss.backward()
        optimizer.step()
        losses.append(loss.detach().cpu().item())
        batch = batch.cpu()
        
    return np.mean(losses)
    
def multiclass_eval(classifier, device, loader, return_pred=False):
    classifier.eval()
    y_true = []
    y_pred = []
    y_scores = []
    losses = []
    loss_fn = nn.CrossEntropyLoss()
    
    for batch in loader:
        gt = batch['y'].to(device)
        with torch.no_grad():
            y = classifier(batch.to(device)) 
            loss = loss_fn(y.float(), gt.long())
        if return_pred:
            y_scores.append(y.detach().cpu())
            y_pred.append(y.argmax(1).detach().cpu())
            y_true.append(gt.detach().cpu())
        losses.append(loss.detach().cpu().item())

    if return_pred:
        y_true = torch.cat(y_true, dim = 0).detach().cpu().numpy()
        y_pred = torch.cat(y_pred, dim = 0).detach().cpu().numpy()
        y_scores = torch.cat(y_scores, dim = 0).detach().cpu().numpy()
        return np.mean(losses), y_true, y_pred, y_scores
    else:
        return np.mean(losses)


In [100]:
invalid_validx = np.where(val_sc.isna())[0]
invalid_trainidx = np.where(train_sc.isna())[0]

device = 'cuda:1'
fc_thr = 0.5
sc_thr = 0.1
f1_list = []
auc_list = []
for brain_code in brain_codes:
    best_f1 = 0
    # p_train_data = all_train_data['gen_img'] # gen_img pred_zs
    p_val_data = val_data['gen_img'] #  gen_img pred_zs
    # p_train_label = train_labelid[:, np.isin(uni_nd, brain_code)]
    p_val_label = val_labelid[:, np.isin(uni_nd, brain_code)]
    # if (p_train_label.sum(1)!=0).sum() < 2: continue
    if (p_val_label.sum(1)!=0).sum() < 2: continue
    print(brain_code)
    
    pp_val_label = torch.zeros(len(p_val_label)).long()
    pp_val_label[p_val_label.sum(1)!=0] = torch.from_numpy(p_val_label.argmax(1)+1)[p_val_label.sum(1)!=0]
    idx0 = np.where(p_val_label.sum(1)!=0)[0].tolist()
    idx = np.where(p_val_label.sum(1)==0)[0].tolist()
    random.shuffle(idx)
    idx = idx[:len(idx0)] + idx0
    pp_val_data = p_val_data[idx]
    pp_val_label = pp_val_label[idx]
    xyedge_datalist = []
    for i in range(len(pp_val_data)):
        if i in invalid_validx.tolist(): 
            sc = pp_val_data[i].numpy()
        else:
            sc = val_sc.iloc[i]
        xyedge_datalist.append(Data(
            x=pp_val_data[i], 
            adj_sc=torch.from_numpy(sc>sc_thr)[None], 
            adj_fc=pp_val_data[i][None]>fc_thr, 
            edge_index=torch.stack(torch.where(pp_val_data[i]>fc_thr)),
            y=pp_val_label[i:i+1]))
    if len(xyedge_datalist) == 0: continue
    val_batches = [Batch.from_data_list(xyedge_datalist[bi:bi+batch_size]) for bi in trange(0, len(xyedge_datalist), batch_size, desc='prepare val batches')]
    
    
    model = GraphNet(in_channel=116, out_channel=2)
    optimizer = optim.Adam(list(model.parameters()), lr=lr, weight_decay=decay)
    # model.load_state_dict(torch.load(f'../neuro_detour/ICD_NeuroDetourLR5e-3noLN_out/ckpt_{brain_code}.ckpt', weights_only=False)['ckpt'])
    model = model.to(device)
    
    pp_train_label = torch.zeros(len(p_train_label)).long()
    pp_train_label[p_train_label.sum(1)!=0] = torch.from_numpy(p_train_label.argmax(1)+1)[p_train_label.sum(1)!=0]
    idx0 = np.where(p_train_label.sum(1)!=0)[0].tolist()
    idx = np.where(p_train_label.sum(1)==0)[0].tolist()
    random.shuffle(idx)
    idx = idx[:len(idx0)] + idx0
    pp_train_data = p_train_data[idx]
    pp_train_label = pp_train_label[idx]
    xyedge_datalist = []
    for i in range(len(pp_train_data)):
        if i in invalid_trainidx.tolist(): 
            sc = pp_train_data[i].numpy()
        else:
            sc = train_sc.iloc[i]
        xyedge_datalist.append(Data(
            x=pp_train_data[i], 
            adj_sc=torch.from_numpy(sc>sc_thr)[None], 
            adj_fc=pp_train_data[i][None]>fc_thr, 
            edge_index=torch.stack(torch.where(pp_train_data[i]>fc_thr)),
            y=pp_train_label[i:i+1]))
    if len(xyedge_datalist) == 0: continue
    train_idx = list(range(len(xyedge_datalist)))
    pbar = trange(1, epochs+1)
    for epoch in pbar:
        random.shuffle(train_idx)
        train_batches = [Batch.from_data_list([xyedge_datalist[idxi] for idxi in train_idx[bi : bi+batch_size]]) for bi in range(0, len(xyedge_datalist), batch_size)]
        
        train_loss = multiclass_train(model, device, train_batches, optimizer, epoch)
    
        # first batch is nan for unknown reasons
        _, gt, preds, scores = multiclass_eval(model, device, [val_batches[0]]+val_batches, return_pred=True) 
        gt = gt[val_batches[0].y.shape[0]:]
        preds = preds[val_batches[0].y.shape[0]:]
        scores = scores[val_batches[0].y.shape[0]:]
        scores = np_softmax(scores)
        # break
        prec, rec, f1, _ = precision_recall_fscore_support(gt, preds, average='weighted')
        fpr, tpr, thresholds = roc_curve(gt, scores[:, 1], pos_label=1)
        auc_area = auc(fpr, tpr)
        pbar.set_description(f'Epoch {epoch}: Trian loss {train_loss:.6f}, F1 {f1} AUC {auc_area}')
        if f1 > best_f1:
            best_f1 = f1
            best_prec = prec
            best_auc_area = auc_area
    f1_list.append(best_f1)
    auc_list.append(best_auc_area)

print(f'F1: {np.mean(f1_list)*100:.2f}'+'$_{\pm'+f'{np.std(f1_list)*100:.2f}'+'}$')
print(f'AUC: {np.mean(auc_list)*100:.2f}'+'$_{\pm'+f'{np.std(auc_list)*100:.2f}'+'}$')


F03
ERROR! Session/line number was not unique in database. History logging moved to new session 725


prepare val batches: 100%|███████████████████████| 1/1 [00:00<00:00, 490.45it/s]


F10


prepare val batches: 100%|███████████████████████| 2/2 [00:00<00:00, 280.12it/s]


G20


...

G31


prepare val batches: 100%|███████████████████████| 1/1 [00:00<00:00, 477.06it/s]


R41


prepare val batches: 100%|███████████████████████| 1/1 [00:00<00:00, 458.85it/s]


F1: 56.53$_{\pm16.78}$
AUC: 59.99$_{\pm9.40}$


In [101]:
import torch.optim as optim
decay = 0

epochs = 30
lr = 0.00001
batch_size = 32
device = 'cuda:1'
fc_thr = 0.5
sc_thr = 0.1
f1_list = []
auc_list = []
for brain_code in heart_codes:
    best_f1 = 0
    # p_train_data = all_train_data['gen_img'] # gen_img pred_zs
    p_val_data = val_data['gen_img'] #  gen_img pred_zs
    # p_train_label = train_labelid[:, np.isin(uni_nd, brain_code)]
    p_val_label = val_labelid[:, np.isin(uni_nd, brain_code)]
    # if (p_train_label.sum(1)!=0).sum() < 2: continue
    if (p_val_label.sum(1)!=0).sum() < 2: continue
    print(brain_code)
    
    pp_val_label = torch.zeros(len(p_val_label)).long()
    pp_val_label[p_val_label.sum(1)!=0] = torch.from_numpy(p_val_label.argmax(1)+1)[p_val_label.sum(1)!=0]
    idx0 = np.where(p_val_label.sum(1)!=0)[0].tolist()
    idx = np.where(p_val_label.sum(1)==0)[0].tolist()
    random.shuffle(idx)
    idx = idx[:len(idx0)] + idx0
    pp_val_data = p_val_data[idx]
    pp_val_label = pp_val_label[idx]
    xyedge_datalist = []
    for i in range(len(pp_val_data)):
        if i in invalid_validx.tolist(): 
            sc = pp_val_data[i].numpy()
        else:
            sc = val_sc.iloc[i]
        xyedge_datalist.append(Data(
            x=pp_val_data[i], 
            adj_sc=torch.from_numpy(sc>sc_thr)[None], 
            adj_fc=pp_val_data[i][None]>fc_thr, 
            edge_index=torch.stack(torch.where(pp_val_data[i]>fc_thr)),
            y=pp_val_label[i:i+1]))
    if len(xyedge_datalist) == 0: continue
    val_batches = [Batch.from_data_list(xyedge_datalist[bi:bi+batch_size]) for bi in range(0, len(xyedge_datalist), batch_size)]
    
    
    model = GraphNet(in_channel=116, out_channel=2)
    optimizer = optim.Adam(list(model.parameters()), lr=lr, weight_decay=decay)
    # model.load_state_dict(torch.load(f'../neuro_detour/ICD_NeuroDetourLR5e-3noLN_out/ckpt_{brain_code}.ckpt', weights_only=False)['ckpt'])
    model = model.to(device)
    
    pp_train_label = torch.zeros(len(p_train_label)).long()
    pp_train_label[p_train_label.sum(1)!=0] = torch.from_numpy(p_train_label.argmax(1)+1)[p_train_label.sum(1)!=0]
    idx0 = np.where(p_train_label.sum(1)!=0)[0].tolist()
    idx = np.where(p_train_label.sum(1)==0)[0].tolist()
    random.shuffle(idx)
    idx = idx[:len(idx0)] + idx0
    pp_train_data = p_train_data[idx]
    pp_train_label = pp_train_label[idx]
    xyedge_datalist = []
    for i in range(len(pp_train_data)):
        if i in invalid_trainidx.tolist(): 
            sc = pp_train_data[i].numpy()
        else:
            sc = train_sc.iloc[i]
        xyedge_datalist.append(Data(
            x=pp_train_data[i], 
            adj_sc=torch.from_numpy(sc>sc_thr)[None], 
            adj_fc=pp_train_data[i][None]>fc_thr, 
            edge_index=torch.stack(torch.where(pp_train_data[i]>fc_thr)),
            y=pp_train_label[i:i+1]))
    if len(xyedge_datalist) == 0: continue
    train_idx = list(range(len(xyedge_datalist)))
    pbar = trange(1, epochs+1)
    for epoch in pbar:
        random.shuffle(train_idx)
        train_batches = [Batch.from_data_list([xyedge_datalist[idxi] for idxi in train_idx[bi : bi+batch_size]]) for bi in range(0, len(xyedge_datalist), batch_size)]
        
        train_loss = multiclass_train(model, device, train_batches, optimizer, epoch)
    
        # first batch is nan for unknown reasons
        _, gt, preds, scores = multiclass_eval(model, device, [val_batches[0]]+val_batches, return_pred=True) 
        gt = gt[val_batches[0].y.shape[0]:]
        preds = preds[val_batches[0].y.shape[0]:]
        scores = scores[val_batches[0].y.shape[0]:]
        scores = np_softmax(scores)
        # break
        prec, rec, f1, _ = precision_recall_fscore_support(gt, preds, average='weighted')
        fpr, tpr, thresholds = roc_curve(gt, scores[:, 1], pos_label=1)
        auc_area = auc(fpr, tpr)
        pbar.set_description(f'Epoch {epoch}: Trian loss {train_loss:.6f}, F1 {f1:.4f} AUC {auc_area:.4f}')
        if f1 > best_f1:
            best_f1 = f1
            best_prec = prec
            best_auc_area = auc_area
    f1_list.append(best_f1)
    auc_list.append(best_auc_area)

print(f'F1: {np.mean(f1_list)*100:.2f}'+'$_{\pm'+f'{np.std(f1_list)*100:.2f}'+'}$')
print(f'AUC: {np.mean(auc_list)*100:.2f}'+'$_{\pm'+f'{np.std(auc_list)*100:.2f}'+'}$')


R00


prepare val batches: 100%|███████████████████████| 2/2 [00:00<00:00, 287.52it/s]
Epoch 30: Trian loss 14.494724, F1 0.3333333333333333 AUC 0.5: 100%|█| 30/30 [00


I48


prepare val batches: 100%|███████████████████████| 6/6 [00:00<00:00, 437.24it/s]
Epoch 30: Trian loss 13.913495, F1 0.446606404167305 AUC 0.4871031746031746: 100


I49


...

I25


prepare val batches: 100%|███████████████████████| 8/8 [00:00<00:00, 472.62it/s]
Epoch 30: Trian loss 14.449964, F1 0.5361907017128423 AUC 0.5299945179732164: 10


R55


prepare val batches: 100%|███████████████████████| 1/1 [00:00<00:00, 419.85it/s]
Epoch 30: Trian loss 14.349194, F1 0.3333333333333333 AUC 0.5432098765432098: 10

F1: 48.96$_{\pm8.04}$
AUC: 49.69$_{\pm12.69}$


In [102]:

device = 'cuda:1'
fc_thr = 0.5
sc_thr = 0.1
f1_list = []
auc_list = []
for brain_code in kidney_codes:
    best_f1 = 0
    # p_train_data = all_train_data['gen_img'] # gen_img pred_zs
    p_val_data = val_data['gen_img'] #  gen_img pred_zs
    # p_train_label = train_labelid[:, np.isin(uni_nd, brain_code)]
    p_val_label = val_labelid[:, np.isin(uni_nd, brain_code)]
    # if (p_train_label.sum(1)!=0).sum() < 2: continue
    if (p_val_label.sum(1)!=0).sum() < 2: continue
    print(brain_code)
    
    pp_val_label = torch.zeros(len(p_val_label)).long()
    pp_val_label[p_val_label.sum(1)!=0] = torch.from_numpy(p_val_label.argmax(1)+1)[p_val_label.sum(1)!=0]
    idx0 = np.where(p_val_label.sum(1)!=0)[0].tolist()
    idx = np.where(p_val_label.sum(1)==0)[0].tolist()
    random.shuffle(idx)
    idx = idx[:len(idx0)] + idx0
    pp_val_data = p_val_data[idx]
    pp_val_label = pp_val_label[idx]
    xyedge_datalist = []
    for i in range(len(pp_val_data)):
        if i in invalid_validx.tolist(): 
            sc = pp_val_data[i].numpy()
        else:
            sc = val_sc.iloc[i]
        xyedge_datalist.append(Data(
            x=pp_val_data[i], 
            adj_sc=torch.from_numpy(sc>sc_thr)[None], 
            adj_fc=pp_val_data[i][None]>fc_thr, 
            edge_index=torch.stack(torch.where(pp_val_data[i]>fc_thr)),
            y=pp_val_label[i:i+1]))
    if len(xyedge_datalist) == 0: continue
    val_batches = [Batch.from_data_list(xyedge_datalist[bi:bi+batch_size]) for bi in range(0, len(xyedge_datalist), batch_size)]
    
    
    model = GraphNet(in_channel=116, out_channel=2)
    optimizer = optim.Adam(list(model.parameters()), lr=lr, weight_decay=decay)
    # model.load_state_dict(torch.load(f'../neuro_detour/ICD_NeuroDetourLR5e-3noLN_out/ckpt_{brain_code}.ckpt', weights_only=False)['ckpt'])
    model = model.to(device)
    
    pp_train_label = torch.zeros(len(p_train_label)).long()
    pp_train_label[p_train_label.sum(1)!=0] = torch.from_numpy(p_train_label.argmax(1)+1)[p_train_label.sum(1)!=0]
    idx0 = np.where(p_train_label.sum(1)!=0)[0].tolist()
    idx = np.where(p_train_label.sum(1)==0)[0].tolist()
    random.shuffle(idx)
    idx = idx[:len(idx0)] + idx0
    pp_train_data = p_train_data[idx]
    pp_train_label = pp_train_label[idx]
    xyedge_datalist = []
    for i in range(len(pp_train_data)):
        if i in invalid_trainidx.tolist(): 
            sc = pp_train_data[i].numpy()
        else:
            sc = train_sc.iloc[i]
        xyedge_datalist.append(Data(
            x=pp_train_data[i], 
            adj_sc=torch.from_numpy(sc>sc_thr)[None], 
            adj_fc=pp_train_data[i][None]>fc_thr, 
            edge_index=torch.stack(torch.where(pp_train_data[i]>fc_thr)),
            y=pp_train_label[i:i+1]))
    if len(xyedge_datalist) == 0: continue
    train_idx = list(range(len(xyedge_datalist)))
    pbar = trange(1, epochs+1)
    for epoch in pbar:
        random.shuffle(train_idx)
        train_batches = [Batch.from_data_list([xyedge_datalist[idxi] for idxi in train_idx[bi : bi+batch_size]]) for bi in range(0, len(xyedge_datalist), batch_size)]
        
        train_loss = multiclass_train(model, device, train_batches, optimizer, epoch)
    
        # first batch is nan for unknown reasons
        _, gt, preds, scores = multiclass_eval(model, device, [val_batches[0]]+val_batches, return_pred=True) 
        gt = gt[val_batches[0].y.shape[0]:]
        preds = preds[val_batches[0].y.shape[0]:]
        scores = scores[val_batches[0].y.shape[0]:]
        scores = np_softmax(scores)
        # break
        prec, rec, f1, _ = precision_recall_fscore_support(gt, preds, average='weighted')
        fpr, tpr, thresholds = roc_curve(gt, scores[:, 1], pos_label=1)
        auc_area = auc(fpr, tpr)
        pbar.set_description(f'Epoch {epoch}: Trian loss {train_loss:.6f}, F1 {f1:.4f} AUC {auc_area:.4f}')
        if f1 > best_f1:
            best_f1 = f1
            best_prec = prec
            best_auc_area = auc_area
    f1_list.append(best_f1)
    auc_list.append(best_auc_area)

print(f'F1: {np.mean(f1_list)*100:.2f}'+'$_{\pm'+f'{np.std(f1_list)*100:.2f}'+'}$')
print(f'AUC: {np.mean(auc_list)*100:.2f}'+'$_{\pm'+f'{np.std(auc_list)*100:.2f}'+'}$')


N02


Epoch 30: Trian loss 14.512403, F1 0.8286 AUC 0.8333: 100%|█| 30/30 [00:09<00:00


N11


Epoch 30: Trian loss 14.553178, F1 0.7333 AUC 0.7500: 100%|█| 30/30 [00:09<00:00


N13


...

N97


Epoch 30: Trian loss 14.193026, F1 0.5152 AUC 0.5781: 100%|█| 30/30 [00:10<00:00


N99


Epoch 30: Trian loss 14.020763, F1 0.3333 AUC 0.5000: 100%|█| 30/30 [00:09<00:00

F1: 54.57$_{\pm13.94}$
AUC: 57.05$_{\pm13.70}$


In [104]:
lr = 1e-3
device = 'cuda:1'
fc_thr = 0.5
sc_thr = 0.1
f1_list = []
auc_list = []
for _ in range(5):
    
    for brain_code in liver_codes:
        best_f1 = 0
        # p_train_data = all_train_data['gen_img'] # gen_img pred_zs
        p_val_data = val_data['gen_img'] #  gen_img pred_zs
        # p_train_label = train_labelid[:, np.isin(uni_nd, brain_code)]
        p_val_label = val_labelid[:, np.isin(uni_nd, brain_code)]
        # if (p_train_label.sum(1)!=0).sum() < 2: continue
        if (p_val_label.sum(1)!=0).sum() < 2: continue
        print(brain_code)
        
        pp_val_label = torch.zeros(len(p_val_label)).long()
        pp_val_label[p_val_label.sum(1)!=0] = torch.from_numpy(p_val_label.argmax(1)+1)[p_val_label.sum(1)!=0]
        idx0 = np.where(p_val_label.sum(1)!=0)[0].tolist()
        idx = np.where(p_val_label.sum(1)==0)[0].tolist()
        random.shuffle(idx)
        idx = idx[:len(idx0)] + idx0
        pp_val_data = p_val_data[idx]
        pp_val_label = pp_val_label[idx]
        xyedge_datalist = []
        for i in range(len(pp_val_data)):
            if i in invalid_validx.tolist(): 
                sc = pp_val_data[i].numpy()
            else:
                sc = val_sc.iloc[i]
            xyedge_datalist.append(Data(
                x=pp_val_data[i], 
                adj_sc=torch.from_numpy(sc>sc_thr)[None], 
                adj_fc=pp_val_data[i][None]>fc_thr, 
                edge_index=torch.stack(torch.where(pp_val_data[i]>fc_thr)),
                y=pp_val_label[i:i+1]))
        if len(xyedge_datalist) == 0: continue
        val_batches = [Batch.from_data_list(xyedge_datalist[bi:bi+batch_size]) for bi in range(0, len(xyedge_datalist), batch_size)]
        
        
        model = GraphNet(in_channel=116, out_channel=2)
        optimizer = optim.Adam(list(model.parameters()), lr=lr, weight_decay=decay)
        # model.load_state_dict(torch.load(f'../neuro_detour/ICD_NeuroDetourLR5e-3noLN_out/ckpt_{brain_code}.ckpt', weights_only=False)['ckpt'])
        model = model.to(device)
        
        pp_train_label = torch.zeros(len(p_train_label)).long()
        pp_train_label[p_train_label.sum(1)!=0] = torch.from_numpy(p_train_label.argmax(1)+1)[p_train_label.sum(1)!=0]
        idx0 = np.where(p_train_label.sum(1)!=0)[0].tolist()
        idx = np.where(p_train_label.sum(1)==0)[0].tolist()
        random.shuffle(idx)
        idx = idx[:len(idx0)] + idx0
        pp_train_data = p_train_data[idx]
        pp_train_label = pp_train_label[idx]
        xyedge_datalist = []
        for i in range(len(pp_train_data)):
            if i in invalid_trainidx.tolist(): 
                sc = pp_train_data[i].numpy()
            else:
                sc = train_sc.iloc[i]
            xyedge_datalist.append(Data(
                x=pp_train_data[i], 
                adj_sc=torch.from_numpy(sc>sc_thr)[None], 
                adj_fc=pp_train_data[i][None]>fc_thr, 
                edge_index=torch.stack(torch.where(pp_train_data[i]>fc_thr)),
                y=pp_train_label[i:i+1]))
        if len(xyedge_datalist) == 0: continue
        train_idx = list(range(len(xyedge_datalist)))
        pbar = trange(1, epochs+1)
        for epoch in pbar:
            random.shuffle(train_idx)
            train_batches = [Batch.from_data_list([xyedge_datalist[idxi] for idxi in train_idx[bi : bi+batch_size]]) for bi in range(0, len(xyedge_datalist), batch_size)]
            
            train_loss = multiclass_train(model, device, train_batches, optimizer, epoch)
        
            # first batch is nan for unknown reasons
            _, gt, preds, scores = multiclass_eval(model, device, [val_batches[0]]+val_batches, return_pred=True) 
            gt = gt[val_batches[0].y.shape[0]:]
            preds = preds[val_batches[0].y.shape[0]:]
            scores = scores[val_batches[0].y.shape[0]:]
            scores = np_softmax(scores)
            # break
            prec, rec, f1, _ = precision_recall_fscore_support(gt, preds, average='weighted')
            fpr, tpr, thresholds = roc_curve(gt, scores[:, 1], pos_label=1)
            auc_area = auc(fpr, tpr)
            pbar.set_description(f'Epoch {epoch}: Trian loss {train_loss:.6f}, F1 {f1:.4f} AUC {auc_area:.4f}')
            if f1 > best_f1:
                best_f1 = f1
                best_prec = prec
                best_auc_area = auc_area
        f1_list.append(best_f1)
        auc_list.append(best_auc_area)

print(f'F1: {np.mean(f1_list)*100:.2f}'+'$_{\pm'+f'{np.std(f1_list)*100:.2f}'+'}$')
print(f'AUC: {np.mean(auc_list)*100:.2f}'+'$_{\pm'+f'{np.std(auc_list)*100:.2f}'+'}$')


K73


Epoch 30: Trian loss 1.258804, F1 0.7333 AUC 0.7500: 100%|█| 30/30 [00:10<00:00,


K74


Epoch 30: Trian loss 1.325383, F1 0.5000 AUC 0.4583: 100%|█| 30/30 [00:09<00:00,


K76


...

K74


Epoch 30: Trian loss 1.362277, F1 0.5000 AUC 0.4444: 100%|█| 30/30 [00:09<00:00,


K76


Epoch 30: Trian loss 1.324091, F1 0.2941 AUC 0.1667: 100%|█| 30/30 [00:09<00:00,

F1: 66.49$_{\pm20.64}$
AUC: 64.72$_{\pm23.11}$


## Transformer

In [2]:
val_data = torch.load('ntp_gen-fc_val.pth')

In [6]:
train_label = torch.load('ntp_text_embed_train.pth')
val_label = torch.load('ntp_text_embed_val.pth')

In [14]:
import numpy as np
train_nd = np.array(train_label['next_disease'])
val_nd = np.array(val_label['next_disease'])
uni_nd = np.unique(np.concatenate([np.unique(train_nd), np.unique(val_nd)]))
train_labelid = np.stack([uni_nd==i for i in train_nd]).astype(float)
val_labelid = np.stack([uni_nd==i for i in val_nd]).astype(float)
uni_nd.shape, train_labelid.shape, val_labelid.shape

((964,), (43049, 964), (10709, 964))

In [7]:
val_data.keys(), train_label.keys(), val_label.keys()

(dict_keys(['pred_zs', 'gen_img']),
 dict_keys(['text_embed', 'data_id', 'next_disease', 'text_cond']),
 dict_keys(['text_embed', 'data_id', 'next_disease', 'text_cond']))

In [9]:
[val_data[k].shape for k in val_data]

[torch.Size([10709, 272, 256]), torch.Size([10709, 116, 116])]

In [24]:
bsz = 256
startbatchi = 0
endbatchi = len(train_labelid)//4 # expect endbatchi
p1 = [[batchi, batchi+bsz] for batchi in range(startbatchi, endbatchi, bsz)][-1][-1] # real endbatchi
print(endbatchi, p1-startbatchi, endbatchi-startbatchi, train_data[0]['pred_zs'].shape)
# sum_endbatchi += endbatchi

bsz = 512
startbatchi = len(train_labelid)//4
endbatchi = len(train_labelid)//4 + len(train_labelid)//4 # expect endbatchi
p1 = [[batchi, batchi+bsz] for batchi in range(startbatchi, endbatchi, bsz)][-1][-1] # real endbatchi
print(endbatchi, p1-startbatchi, endbatchi-startbatchi, train_data[1]['pred_zs'].shape)

startbatchi = len(train_labelid)//4 + len(train_labelid)//4 # expect endbatchi
endbatchi = len(train_labelid)//4 + len(train_labelid)//4 + len(train_labelid)//4
p1 = [[batchi, batchi+bsz] for batchi in range(startbatchi, endbatchi, bsz)][-1][-1] # real endbatchi
print(endbatchi, p1-startbatchi, endbatchi-startbatchi, train_data[2]['pred_zs'].shape)

startbatchi = len(train_labelid)//4 + len(train_labelid)//4 + len(train_labelid)//4
endbatchi = len(train_labelid)
print(endbatchi, endbatchi-startbatchi, train_data[3]['pred_zs'].shape)


10762 11008 10762 torch.Size([11008, 272, 256])
21524 11264 10762 torch.Size([11264, 272, 256])
32286 11264 10762 torch.Size([11264, 272, 256])
43049 10763 torch.Size([10763, 272, 256])


In [79]:
train_data[-1].keys()

dict_keys(['pred_zs', 'gen_img'])

In [3]:

train_data = [
    torch.load('ntp_gen-fc_train0.pth'),
    torch.load('ntp_gen-fc_train1.pth'),
    torch.load('ntp_gen-fc_train2.pth'),
    torch.load('ntp_gen-fc_train3.pth'),
]


In [20]:
train_data[-1]['pred_zs'].shape

torch.Size([10763, 272, 256])

In [26]:
all_train_data = {k: torch.cat([
    train_data[0][k][:10762], 
    train_data[1][k][:10762], 
    train_data[2][k][:10762], 
    train_data[3][k]]) for k in train_data[0]}
[all_train_data[k].shape for k in all_train_data]

[torch.Size([43049, 272, 256]), torch.Size([43049, 116, 116])]

In [81]:
# source: https://www.medrxiv.org/content/medrxiv/early/2022/02/17/2022.02.16.22271085/DC1/embed/media-1.pdf?download=true
brain_code_str='''A81 
E71, E75, E75, E75 
F01, F02, F03, F04, F10, F10,
F10, F10, F13, F13, F13,
F13, F18, F18
G10, G20, G30, G23, G31, G31
R41'''
brain_codes = []
for line in brain_code_str.split('\n'):
    brain_codes.extend([i for i in line.split(', ')])
brain_codes = np.unique(brain_codes).tolist()
print(brain_codes)
# source: https://www.cms.gov/medicare/coding/icd10/downloads/icd10clinicalconceptscardiology1.pdf
heart_codes = ['R00', 'I48', 'I49', 'I20', 'R07', 'I34', 'I35', 'I10', 'I50', 'I21', 'I22', 'I23', 'I25', 'R55']
kidney_codes = [f'N{i:02d}' for i in range(100)]
liver_codes = [f'K{i:02d}' for i in range(70, 78, 1)]

# brain_codes = [tokenizer[c.item()] for c in brain_codes if c.item() in tokenizer]
# heart_codes = [tokenizer[c] for c in heart_codes if c in tokenizer]
# kidney_codes = [tokenizer[c] for c in kidney_codes if c in tokenizer]
# liver_codes = [tokenizer[c] for c in liver_codes if c in tokenizer]

['A81 ', 'E71', 'E75', 'E75 ', 'F01', 'F02', 'F03', 'F04', 'F10', 'F10,', 'F13', 'F13,', 'F18', 'G10', 'G20', 'G23', 'G30', 'G31', 'R41']


In [76]:
val_data['pred_zs'].shape, all_train_data['pred_zs'].shape, train_labelid.shape, val_labelid.shape, val_labelid.max(), train_labelid.max()

(torch.Size([10709, 272, 256]),
 torch.Size([43049, 272, 256]),
 (43049, 964),
 (10709, 964),
 np.float64(1.0),
 np.float64(1.0))

In [86]:
np.isin(train_label['next_disease'], brain_codes).sum(), np.isin(val_label['next_disease'], brain_codes).sum()

(np.int64(184), np.int64(43))

In [98]:

p_train_label.shape, (p_train_label==1).sum(1), p_val_label.shape, (p_val_label==1).sum(1)

((184, 9),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1]),
 (43, 9),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [156]:
p_train_data = all_train_data['pred_zs'] # gen_img pred_zs
p_val_data = val_data['pred_zs'] #  gen_img pred_zs
p_train_label = train_labelid[:, np.isin(uni_nd, brain_codes)]
p_val_label = val_labelid[:, np.isin(uni_nd, brain_codes)]
pp_train_label = torch.zeros(len(p_train_label)).long()
pp_train_label[p_train_label.sum(1)!=0] = torch.from_numpy(p_train_label.argmax(1))[p_train_label.sum(1)!=0]
pp_val_label = torch.zeros(len(p_val_label)).long()
pp_val_label[p_val_label.sum(1)!=0] = torch.from_numpy(p_val_label.argmax(1))[p_val_label.sum(1)!=0]
import random 
idx0 = np.where(p_train_label.sum(1)!=0)[0].tolist()
idx = np.where(p_train_label.sum(1)==0)[0].tolist()
random.shuffle(idx)
idx = idx[:len(idx0)] + idx0
pp_train_data = p_train_data[idx]
pp_train_label = pp_train_label[idx]
idx0 = np.where(p_val_label.sum(1)!=0)[0].tolist()
idx = np.where(p_val_label.sum(1)==0)[0].tolist()
random.shuffle(idx)
idx = idx[:len(idx0)] + idx0
pp_val_data = p_val_data[idx]
pp_val_label = pp_val_label[idx]
print(np.bincount(pp_train_label), np.bincount(pp_val_label))

[188   7   1 106  22   1   6  13  24] [43  0  2 29  6  0  0  3  3]


In [ ]:
BrainMass-MLP:
Heart
